# Running JarvAIs on breast cancer clinical and radiomic data

In [1]:
%cd ../..

/home/bhkuser/bhklab/katy/jarvais-breast


In [2]:
from jarvais.analyzer import Analyzer

from damply import dirs
import pandas as pd

/home/bhkuser/bhklab/katy/jarvais-breast/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data_dir = dirs.PROCDATA / "Breast" / "TCIA_ISPY2" / "BRADCURE_analysis"
output_dir = dirs.RESULTS / "TCIA_ISPY2" / "jarvais"

if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)

In [4]:
clinical_ISPY2 = pd.read_csv(data_dir / "clinical_ISPY2_combined.csv")
clinical_ISPY2 = clinical_ISPY2.drop(['ethnicity', 'Race'], axis=1)

radiomics_ISPY2 = pd.read_csv(data_dir / "features_only_ISPY2_combined.csv")

data_ISPY2 = pd.merge(clinical_ISPY2, radiomics_ISPY2, left_on="Patient_ID", right_on="Patient_ID")
data_ISPY2 = data_ISPY2.drop('Patient_ID', axis=1)

In [5]:
data_ISPY2.head()

,Arm,HR,HER2,MP,pCR,Age_at_Screening,menopausal_status,original_shape_MeshVolume,original_shape_VoxelVolume,original_shape_SurfaceArea,...,lbp-2D_gldm_LargeDependenceLowGrayLevelEmphasis,lbp-2D_gldm_LowGrayLevelEmphasis,lbp-2D_gldm_SmallDependenceEmphasis,lbp-2D_gldm_SmallDependenceHighGrayLevelEmphasis,lbp-2D_gldm_SmallDependenceLowGrayLevelEmphasis,lbp-2D_ngtdm_Busyness,lbp-2D_ngtdm_Coarseness,lbp-2D_ngtdm_Complexity,lbp-2D_ngtdm_Contrast,lbp-2D_ngtdm_Strength
0,Paclitaxel + Ganitumab,0,0,1,0,53.0,Postmenopausal (prior bilateral ovariectomy OR...,351.958333,464.0,999.157583,...,81.301724,1.0,0.038052,0.038052,0.038052,0.0,1000000.0,0.0,0.0,0.0
1,Paclitaxel + Ganetespib,1,0,1,1,32.0,Premenopausal(<6 months since LMP AND no prior...,74.125000,110.0,235.037999,...,75.836364,1.0,0.047906,0.047906,0.047906,0.0,1000000.0,0.0,0.0,0.0
2,Paclitaxel + MK-2206,0,0,0,0,26.0,Premenopausal(<6 months since LMP AND no prior...,42195.875000,42423.0,31347.316504,...,479.180798,1.0,0.004241,0.004241,0.004241,0.0,1000000.0,0.0,0.0,0.0
3,Paclitaxel + Pembrolizumab,1,0,0,0,59.0,Postmenopausal (prior bilateral ovariectomy OR...,2714.916667,2998.0,4546.183710,...,251.168779,1.0,0.010436,0.010436,0.010436,0.0,1000000.0,0.0,0.0,0.0
4,Paclitaxel,1,0,0,0,51.0,Perimenopausal(6-12 months since LMP AND no pr...,18251.458333,18547.0,11716.978631,...,509.792150,1.0,0.004529,0.004529,0.004529,0.0,1000000.0,0.0,0.0,0.0


In [6]:
analyzer = Analyzer(
    data_ISPY2,
    output_dir = output_dir / "analyzer_outputs",
    categorical_columns = ['Arm', 'HR', 'HER2', 'MP', 'pCR', 'menopausal_status'],
    target_variable = 'pCR',
    task="classification"
)

analyzer.settings.visualization.plots.remove('multiplot')

analyzer.run()

11:41:00 [warning  ] Continuous columns not specified. Inferring from remaining columns. [jarvais] call=analyzer.__init__:81


+------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------+-----------+--------------------------------+
|                                                                  |                                                                                                           | Missing   | Overall                        |
+==================================================================+===========================================================================================================+===========+================================+
| n                                                                |                                                                                                           |           | 985                            |
+------------------------------------------------------------------+--------------------------------------------

11:42:05 [warning  ] Multiplots directory not found: /home/bhkuser/bhklab/katy/jarvais-breast/data/results/TCIA_ISPY2/jarvais/analyzer_outputs/figures/multiplots [jarvais] call=statistical_ranking.find_top_multiplots:70
         [warning  ] No significant results found for dashboard plot. Skipping dashboard image generation. [jarvais] call=dashboard.__call__:95
Font MPDFAA+Inter28ptBold is missing the following glyphs: '
' (\n)


Error in callback <function _draw_all_if_interactive at 0x7fd5cb8b4c20> (for post_execute), with arguments args (),kwargs {}:


MemoryError: std::bad_alloc

MemoryError: std::bad_alloc

<Figure size 610344x508620 with 2 Axes>

In [23]:
from jarvais.trainer import TrainerSupervised

trainer = TrainerSupervised(
    output_dir= output_dir / "trainer",
    target_variable = 'pCR',
    task = 'binary',
    k_folds=5,
    reduction_method='mrmr',
    keep_k=50,
    explain=True
)

print(trainer)

11:15:59 [warning  ] One-hot encoding is disabled for binary and multiclass tasks due to autogluon's OneHotEncoder implementation. If you want to use one-hot encoding, edit the trainer settings manually. [jarvais] call=trainer.__init__:54


TrainerSupervised(settings={
  "output_dir": "/home/bhkuser/bhklab/katy/jarvais-breast/data/results/TCIA_ISPY2/jarvais/trainer",
  "target_variable": "pCR",
  "task": "binary",
  "stratify_on": null,
  "test_size": 0.2,
  "random_state": 42,
  "explain": true,
  "encoding_module": {
    "columns": null,
    "prefix_sep": "|",
    "enabled": false
  },
  "reduction_module": {
    "method": "mrmr",
    "task": "binary",
    "keep_k": 50,
    "enabled": true
  },
  "trainer_module": {
    "output_dir": "/home/bhkuser/bhklab/katy/jarvais-breast/data/results/TCIA_ISPY2/jarvais/trainer",
    "target_variable": "pCR",
    "task": "binary",
    "eval_metric": "roc_auc",
    "k_folds": 5,
    "extra_metrics": [
      "f1",
      "auprc"
    ],
    "kwargs": {}
  }
})


In [24]:
analyzer.data['pCR'] = analyzer.data['pCR'].astype(int)

trainer.run(analyzer.data)

11:16:50 [warning  ] One-hot encoding is disabled.  [jarvais] call=encoding.__call__:34
100%|██████████| 50/50 [00:11<00:00,  4.31it/s]


AttributeError: module 'psutil' has no attribute 'virtual_memory'